# 00: Setup AOI and configuration

**Why:** This notebook creates the single Drive handoff used by every Phase 0 notebook. It streams the USGS High Plains boundary directly to Drive when needed, serializes the AOI, and creates the complete output tree.

- Source: USGS High Plains Aquifer boundary
- Outputs: `00_setup/ogallala_boundary.geojson`, `ogallala_bbox.json`, `aoi_ee_geometry.json`, `config.yaml`, and `utils.py`
- Author: `<name / institution>`

In [1]:
# Local bootstrap. Installs are quiet so the notebook stays readable.
%pip -q install earthengine-api xee xarray netCDF4 h5netcdf geopandas rioxarray regionmask "dask[diagnostics]" requests tqdm python-dotenv pyyaml matplotlib pandas pyproj shapely compliance-checker
from pathlib import Path
import os, sys, json, time, logging, traceback
from datetime import datetime

PROJECT_ID = os.getenv("GEE_PROJECT", "ee-ishansinhagzb")
import ee
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

# Local files hold configuration, logs, and task metadata. GeoTIFF outputs go to Google Drive via EE.
DRIVE_ROOT = Path(os.getenv("OGALLALA_PHASE0_ROOT", "./Ogallala_Phase0"))
SETUP_DIR = DRIVE_ROOT / '00_setup'
SETUP_DIR.mkdir(parents=True, exist_ok=True)
UTILS_SOURCE = '"""Shared utilities for the Ogallala Phase 0 Colab notebooks."""\nfrom __future__ import annotations\n\nimport json\nimport logging\nimport math\nimport os\nimport time\nimport traceback\nfrom datetime import datetime, timezone\nfrom functools import wraps\nfrom pathlib import Path\nfrom typing import Any, Callable, Iterable\n\nimport numpy as np\nimport pandas as pd\n\nREGISTRY_COLUMNS = [\n    "timestamp", "notebook_id", "dataset", "task_id_or_file", "pathway",\n    "status", "start_time", "end_time", "n_bytes", "error_message", "retries",\n]\nTERMINAL = {"COMPLETED", "FAILED"}\n\n\ndef utc_now() -> str:\n    """Return an ISO-8601 UTC timestamp."""\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef ensure_layout(root: Path, leaf: str) -> Path:\n    """Create the standard Drive tree and return one dataset leaf."""\n    root = Path(root)\n    for path in [root / "00_setup", root / "logs", root / "manifests"]:\n        path.mkdir(parents=True, exist_ok=True)\n    for name in ["raw", "cf", "qc"]:\n        (root / leaf / name).mkdir(parents=True, exist_ok=True)\n    registry = root / "logs" / "task_registry.csv"\n    if not registry.exists():\n        pd.DataFrame(columns=REGISTRY_COLUMNS).to_csv(registry, index=False)\n    return root / leaf\n\n\ndef configure_logger(notebook_id: str, root: Path) -> tuple[logging.Logger, Path]:\n    """Configure rotating file and stdout logging for a notebook."""\n    from logging.handlers import RotatingFileHandler\n    log_dir = Path(root) / "logs"\n    log_dir.mkdir(parents=True, exist_ok=True)\n    path = log_dir / f"{notebook_id}_{datetime.now().strftime(\'%Y%m%dT%H%M%S\')}.log"\n    logger = logging.getLogger(notebook_id)\n    logger.setLevel(logging.INFO)\n    logger.handlers.clear()\n    formatter = logging.Formatter("%(asctime)s %(levelname)s %(message)s")\n    file_handler = RotatingFileHandler(path, maxBytes=50 * 1024 * 1024, backupCount=3)\n    file_handler.setFormatter(formatter)\n    stream_handler = logging.StreamHandler()\n    stream_handler.setFormatter(formatter)\n    logger.addHandler(file_handler)\n    logger.addHandler(stream_handler)\n    return logger, path\n\n\ndef registry_path(root: Path) -> Path:\n    """Return the shared task registry path."""\n    path = Path(root) / "logs" / "task_registry.csv"\n    path.parent.mkdir(parents=True, exist_ok=True)\n    if not path.exists():\n        pd.DataFrame(columns=REGISTRY_COLUMNS).to_csv(path, index=False)\n    return path\n\n\ndef append_registry(root: Path, **values: Any) -> None:\n    """Append a normalized task or file event to the registry."""\n    row = {column: values.get(column, "") for column in REGISTRY_COLUMNS}\n    row["timestamp"] = row["timestamp"] or utc_now()\n    path = registry_path(root)\n    frame = pd.DataFrame([row], columns=REGISTRY_COLUMNS)\n    frame.to_csv(path, mode="a", header=path.stat().st_size == 0, index=False)\n\n\ndef logged_step(notebook_id: str, dataset: str, root: Path, logger: logging.Logger) -> Callable:\n    """Decorate an I/O step so failures are logged and recorded."""\n    def decorator(function: Callable) -> Callable:\n        @wraps(function)\n        def wrapper(*args: Any, **kwargs: Any) -> Any:\n            start = utc_now()\n            try:\n                result = function(*args, **kwargs)\n                append_registry(root, notebook_id=notebook_id, dataset=dataset,\n                                task_id_or_file=function.__name__, pathway="xee",\n                                status="COMPLETED", start_time=start, end_time=utc_now())\n                return result\n            except Exception as exc:\n                logger.error("%s failed: %s\\n%s", function.__name__, exc, traceback.format_exc())\n                append_registry(root, notebook_id=notebook_id, dataset=dataset,\n                                task_id_or_file=function.__name__, pathway="xee",\n                                status="FAILED", start_time=start, end_time=utc_now(),\n                                error_message=str(exc))\n                return None\n        return wrapper\n    return decorator\n\n\nclass BatchManager:\n    """Submit non-blocking Earth Engine exports under a small concurrency cap."""\n\n    def __init__(self, root: Path, notebook_id: str, dataset: str,\n                 logger: logging.Logger, max_concurrent: int = 20,\n                 max_daily_tasks: int = 2500) -> None:\n        self.root = Path(root)\n        self.notebook_id = notebook_id\n        self.dataset = dataset\n        self.logger = logger\n        self.max_concurrent = max_concurrent\n        self.max_daily_tasks = max_daily_tasks\n\n    def active_count(self) -> int:\n        """Count locally registered active submissions."""\n        frame = pd.read_csv(registry_path(self.root))\n        if frame.empty:\n            return 0\n        return int(frame["status"].isin(["STARTED", "RUNNING"]).sum())\n\n    def wait_for_slot(self) -> None:\n        """Wait before submitting when the local active cap is reached."""\n        while self.active_count() >= self.max_concurrent:\n            self.logger.info("Active cap reached; sleeping 60 seconds")\n            time.sleep(60)\n\n    def start(self, task: Any, description: str, retries: int = 3) -> str:\n        """Start a task with retry backoff and return its Earth Engine ID."""\n        self.wait_for_slot()\n        delays = [10, 60, 300]\n        start_time = utc_now()\n        for attempt in range(retries + 1):\n            try:\n                task.start()\n                task_id = getattr(task, "id", "") or task.status().get("id", "")\n                append_registry(self.root, notebook_id=self.notebook_id, dataset=self.dataset,\n                                task_id_or_file=task_id, pathway="batch", status="STARTED",\n                                start_time=start_time, retries=attempt)\n                time.sleep(2)\n                return task_id\n            except Exception as exc:\n                message = str(exc).lower()\n                retryable = "429" in message or "rate limit" in message or "quota" in message\n                if not retryable or attempt >= retries:\n                    append_registry(self.root, notebook_id=self.notebook_id, dataset=self.dataset,\n                                    task_id_or_file=description, pathway="batch", status="FAILED",\n                                    start_time=start_time, end_time=utc_now(),\n                                    error_message=str(exc), retries=attempt)\n                    self.logger.error("Export failed: %s", exc)\n                    return ""\n                delay = delays[min(attempt, len(delays) - 1)]\n                self.logger.warning("Rate limit on %s; retrying in %ss", description, delay)\n                append_registry(self.root, notebook_id=self.notebook_id, dataset=self.dataset,\n                                task_id_or_file=description, pathway="batch", status="RETRIED",\n                                start_time=start_time, retries=attempt + 1,\n                                error_message=str(exc))\n                time.sleep(delay)\n        return ""\n\n    def submit_image(self, image: Any, description: str, folder: str,\n                     prefix: str, region: Any, scale: int, crs: str) -> str:\n        """Create and start a Drive GeoTIFF export without polling."""\n        import ee\n        task = ee.batch.Export.image.toDrive(\n            image=image, description=description, folder=folder,\n            fileNamePrefix=prefix, region=region, scale=scale, crs=crs,\n            maxPixels=1e13, fileFormat="GeoTIFF",\n        )\n        return self.start(task, description)\n\n    def monitor_once(self) -> pd.DataFrame:\n        """Poll registered Earth Engine tasks once and update their statuses."""\n        import ee\n        path = registry_path(self.root)\n        frame = pd.read_csv(path)\n        if frame.empty:\n            return frame\n        for index, row in frame.iterrows():\n            if row["status"] not in ["STARTED", "RUNNING"] or not row["task_id_or_file"]:\n                continue\n            try:\n                status = ee.data.getTaskStatus(str(row["task_id_or_file"]))[0]\n                state = status.get("state", "UNKNOWN")\n                mapped = {"READY": "RUNNING", "RUNNING": "RUNNING",\n                          "COMPLETED": "COMPLETED", "FAILED": "FAILED",\n                          "CANCELLED": "FAILED"}.get(state, state)\n                frame.loc[index, "status"] = mapped\n                frame.loc[index, "end_time"] = utc_now() if mapped in TERMINAL else ""\n                frame.loc[index, "error_message"] = status.get("error_message", "")\n            except Exception as exc:\n                self.logger.warning("Could not poll %s: %s", row["task_id_or_file"], exc)\n        frame.to_csv(path, index=False)\n        return frame\n\n\ndef normalize_projected_dataset(ds: Any, crs: str = "EPSG:5070") -> Any:\n    """Normalize raster dimensions and add projected CF coordinates."""\n    import xarray as xr\n    if "lat" in ds.dims and "y" not in ds.dims:\n        ds = ds.rename({"lat": "y"})\n    if "lon" in ds.dims and "x" not in ds.dims:\n        ds = ds.rename({"lon": "x"})\n    if "y" not in ds.coords:\n        ds = ds.assign_coords(y=np.arange(ds.sizes.get("y", 1), dtype=float))\n    if "x" not in ds.coords:\n        ds = ds.assign_coords(x=np.arange(ds.sizes.get("x", 1), dtype=float))\n    ds["y"].attrs.update({"standard_name": "projection_y_coordinate", "units": "m", "axis": "Y"})\n    ds["x"].attrs.update({"standard_name": "projection_x_coordinate", "units": "m", "axis": "X"})\n    if "lat" not in ds:\n        ds["lat"] = xr.DataArray(np.broadcast_to(ds["y"].values[:, None],\n                                                  (ds.sizes["y"], ds.sizes["x"])), dims=("y", "x"))\n    if "lon" not in ds:\n        ds["lon"] = xr.DataArray(np.broadcast_to(ds["x"].values[None, :],\n                                                  (ds.sizes["y"], ds.sizes["x"])), dims=("y", "x"))\n    ds["lat"].attrs.update({"standard_name": "latitude", "units": "degrees_north"})\n    ds["lon"].attrs.update({"standard_name": "longitude", "units": "degrees_east"})\n    return ds\n\n\ndef write_cf_netcdf(ds: Any, path: Path, attrs: dict[str, Any],\n                    variable_attrs: dict[str, dict[str, Any]] | None = None) -> Path:\n    """Write a compressed projected NetCDF with the Phase 0 CF contract."""\n    import xarray as xr\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    ds = normalize_projected_dataset(ds)\n    ds.attrs.update({\n        "Conventions": "CF-1.8", "title": attrs.get("title", "Ogallala Phase 0 dataset"),\n        "history": f"{utc_now()} notebook={attrs.get(\'notebook_id\', \'unknown\')} git_commit=placeholder",\n        "institution": attrs.get("institution", "Ogallala Phase 0 research team"),\n        "source": attrs.get("source", "Google Earth Engine"),\n        "gee_asset_id": attrs.get("gee_asset_id", ""),\n        "references": attrs.get("references", "Groundwater Buffering of Ecosystem Function During Drought Phase 0 plan"),\n        "comment": attrs.get("comment", "Master grid: EPSG:5070; native bands and QA bands preserved."),\n        "featureType": attrs.get("featureType", "grid"),\n    })\n    ds["crs"] = xr.DataArray(0, attrs={\n        "grid_mapping_name": "albers_conical_equal_area", "semi_major_axis": 6378137.0,\n        "inverse_flattening": 298.257222101, "false_easting": 0.0, "false_northing": 0.0,\n        "spatial_ref": crs_wkt(),\n    })\n    variable_attrs = variable_attrs or {}\n    for name, variable in ds.data_vars.items():\n        if name == "crs":\n            continue\n        defaults = {"long_name": name, "units": "1", "_FillValue": -9999.0,\n                    "grid_mapping": "crs", "coordinates": "lat lon"}\n        defaults.update(variable_attrs.get(name, {}))\n        variable.attrs.update(defaults)\n    encoding: dict[str, dict[str, Any]] = {}\n    for name, variable in ds.data_vars.items():\n        if name == "crs":\n            continue\n        fill = variable.attrs.get("_FillValue", -9999.0)\n        encoding[name] = {"zlib": True, "complevel": 4, "_FillValue": fill}\n        if "time" in variable.dims:\n            encoding[name]["chunksizes"] = tuple(min(24, ds.sizes[d]) for d in variable.dims)\n    if "time" in ds.coords:\n        ds["time"].attrs.update({"standard_name": "time", "long_name": "time",\n                                  "units": "days since 1970-01-01 00:00:00",\n                                  "calendar": "proleptic_gregorian", "axis": "T"})\n        encoding["time"] = {"units": "days since 1970-01-01 00:00:00",\n                             "calendar": "proleptic_gregorian"}\n    ds.to_netcdf(path, format="NETCDF4_CLASSIC", engine="netcdf4", encoding=encoding)\n    return path\n\n\ndef crs_wkt() -> str:\n    """Return a WKT representation for the master Albers grid."""\n    try:\n        from pyproj import CRS\n        return CRS.from_epsg(5070).to_wkt()\n    except Exception:\n        return "EPSG:5070"\n\n\ndef qc_netcdf(path: Path, qc_dir: Path) -> dict[str, Any]:\n    """Create summary statistics and a quicklook for a NetCDF file."""\n    import matplotlib.pyplot as plt\n    import xarray as xr\n    path, qc_dir = Path(path), Path(qc_dir)\n    qc_dir.mkdir(parents=True, exist_ok=True)\n    ds = xr.open_dataset(path)\n    summary: dict[str, Any] = {}\n    for name, value in ds.data_vars.items():\n        if name == "crs":\n            continue\n        array = value.values.astype(float)\n        finite = array[np.isfinite(array)]\n        summary[name] = {"min": float(np.min(finite)) if finite.size else None,\n                         "max": float(np.max(finite)) if finite.size else None,\n                         "mean": float(np.mean(finite)) if finite.size else None,\n                         "std": float(np.std(finite)) if finite.size else None,\n                         "n_valid": int(finite.size)}\n        if finite.size:\n            image = np.nanmean(array, axis=0) if "time" in value.dims else array\n            plt.imsave(qc_dir / f"{path.stem}_{name}.png", image, cmap="viridis")\n    summary_path = qc_dir / f"{path.stem}_summary_stats.json"\n    summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")\n    ds.close()\n    return summary\n\n\ndef write_manifest(root: Path, notebook_id: str, files: Iterable[Path],\n                   dataset: str, status: str = "COMPLETED") -> Path:\n    """Write provenance metadata for files produced by one notebook."""\n    entries = []\n    for file_path in files:\n        file_path = Path(file_path)\n        if file_path.exists():\n            entries.append({"file": str(file_path), "size_bytes": file_path.stat().st_size,\n                            "cf_version": "CF-1.8", "dataset": dataset})\n    path = Path(root) / "manifests" / f"{notebook_id}_manifest.json"\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps({"notebook_id": notebook_id, "status": status,\n                                "files": entries}, indent=2), encoding="utf-8")\n    return path\n'
(SETUP_DIR / 'utils.py').write_text(UTILS_SOURCE, encoding='utf-8')
sys.path.insert(0, str(SETUP_DIR))
from utils import *
NOTEBOOK_ID = '00_setup'


Note: you may need to restart the kernel to use updated packages.


In [2]:
import geopandas as gpd
import requests
import zipfile
import io
import yaml

source_dir = SETUP_DIR / 'source_boundary'
shapefile = next(source_dir.glob('*.shp'), None) if source_dir.exists() else None
if shapefile is None:
    url = os.getenv('OGALLALA_BOUNDARY_ZIP_URL', 'https://water.usgs.gov/GIS/dsdl/ds543.zip')
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    source_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
        archive.extractall(source_dir)
    shapefile = next(source_dir.glob('*.shp'))

gdf = gpd.read_file(shapefile).to_crs('EPSG:4326')
gdf.to_file(SETUP_DIR / 'ogallala_boundary.geojson', driver='GeoJSON')
geometry = json.loads((SETUP_DIR / 'ogallala_boundary.geojson').read_text())['features'][0]['geometry']
aoi = ee.Geometry(geometry)
(SETUP_DIR / 'aoi_ee_geometry.json').write_text(json.dumps(aoi.getInfo()), encoding='utf-8')
minx, miny, maxx, maxy = gdf.total_bounds
(SETUP_DIR / 'ogallala_bbox.json').write_text(json.dumps({'west': minx, 'south': miny, 'east': maxx, 'north': maxy}, indent=2), encoding='utf-8')

CONFIG_YAML = '{"DRY_RUN": false, "GEE_PROJECT": "ee-ishansinhagzb", "MASTER_CRS": "EPSG:5070", "MASTER_SCALE": {"01a": 1000, "01b": 1000, "01d": 1000, "01e": 1000, "01f": 1000, "01g": 1000, "01h": 1000, "01i": 1000, "01j": 30, "01k": 1000, "01l": 1000, "01m": 30, "01n": 30, "01o": 30, "01p": 1000, "01q": 1000, "01r": 1000, "01s": 1000}, "AOI_BUFFER_METERS": 1000, "START_DATE": "2000-01-01", "END_DATE": "2024-09-30", "datasets": {"01a": {"name": "GRACE / GRACE-FO Mascons", "asset_id": "NASA/GRACE/MASS_GRIDS_V04/MASCON", "vars": "TWSA", "bands": ["lwe_thickness"], "scale": 1000, "kind": "collection", "start": "2002-04-01", "end": "2024-09-30", "folder": "01a_grace_mascons", "usage": "Basin-scale storage constraint and slow driver; use for closure and screening.", "drive_subfolder": "01_gee_rasters/01a_grace_mascons"}, "01b": {"name": "SMAP L4 Root Zone Soil Moisture and L3 VOD", "asset_id": ["NASA/SMAP/SPL4SMGP/008", "NASA/SMAP/SPL3SMP_E/005"], "vars": "MULTI", "bands": ["sm_rootzone", "vegetation_water_content"], "scale": 1000, "kind": "collection_group", "start": "2015-04-01", "end": "2024-09-30", "folder": "01b_smap_l4_rzsm", "usage": "Climate control and vegetation water signal for dry-down decoupling analysis.", "drive_subfolder": "01_gee_rasters/01b_smap_l4_rzsm"}, "01d": {"name": "TROPOMI SIF", "asset_id": "projects/sat-io/open-datasets/TROPOSIF", "vars": "SIF", "bands": ["SIF"], "scale": 1000, "kind": "collection", "start": "2018-05-01", "end": "2024-09-30", "folder": "01d_tropomi_sif", "usage": "Independent carbon signal for recent-period decoupling analysis.", "drive_subfolder": "01_gee_rasters/01d_tropomi_sif"}, "01e": {"name": "ERA5-Land", "asset_id": "ECMWF/ERA5_LAND/HOURLY", "vars": "MULTI", "bands": ["total_precipitation", "temperature_2m", "dewpoint_temperature_2m", "surface_net_solar_radiation"], "scale": 1000, "kind": "collection", "start": "2002-01-01", "end": "2024-09-30", "folder": "01e_era5_land", "usage": "Meteorological controls for expected-function matching.", "drive_subfolder": "01_gee_rasters/01e_era5_land"}, "01f": {"name": "gridMET", "asset_id": "IDAHO_EPSCOR/GridMET", "vars": "MULTI", "bands": ["pr", "vpd", "etr", "tmmx", "tmmn", "vs"], "scale": 1000, "kind": "collection", "start": "1979-01-01", "end": "2024-09-30", "folder": "01f_gridmet", "usage": "Primary climate grid for inference and matching.", "drive_subfolder": "01_gee_rasters/01f_gridmet"}, "01g": {"name": "MODIS NDVI / EVI", "asset_id": "MODIS/061/MOD13A2", "vars": "MULTI", "bands": ["NDVI", "EVI", "DetailedQA", "SummaryQA", "pixel_reliability"], "scale": 1000, "kind": "collection", "start": "2000-02-18", "end": "2024-09-30", "folder": "01g_modis_ndvi_evi", "usage": "Long-record context, screening, and reversibility testing.", "drive_subfolder": "01_gee_rasters/01g_modis_ndvi_evi"}, "01h": {"name": "MODIS Land Cover", "asset_id": "MODIS/061/MCD12Q1", "vars": "MULTI", "bands": ["LC_Type1", "LC_Prop1_Assessment", "LC_Prop1"], "scale": 1000, "kind": "collection", "start": "2001-01-01", "end": "2023-12-31", "folder": "01h_modis_landcover", "usage": "Vegetation-type and crop versus natural stratification.", "drive_subfolder": "01_gee_rasters/01h_modis_landcover"}, "01i": {"name": "SRTM DEM", "asset_id": "USGS/SRTMGL1_003", "vars": "ELEV", "bands": ["elevation"], "scale": 1000, "kind": "image", "start": "2000-01-01", "end": "2000-12-31", "folder": "01i_srtm_dem", "usage": "Topographic-position stratification.", "drive_subfolder": "01_gee_rasters/01i_srtm_dem"}, "01j": {"name": "USDA CDL", "asset_id": "USDA/NASS/CDL", "vars": "CROP", "bands": ["cropland"], "scale": 30, "kind": "collection", "start": "2008-01-01", "end": "2024-12-31", "folder": "01j_usda_cdl", "usage": "Crop type and irrigation-control stratification.", "drive_subfolder": "01_gee_rasters/01j_usda_cdl"}, "01k": {"name": "GFSAD", "asset_id": "USGS/GFSAD/GLASS/CropMasks/2000", "vars": "IRRIG", "bands": ["landcover"], "scale": 1000, "kind": "image", "start": "2000-01-01", "end": "2000-12-31", "folder": "01k_gfsad", "usage": "Managed versus rainfed area control.", "drive_subfolder": "01_gee_rasters/01k_gfsad"}, "01l": {"name": "Aridity Index", "asset_id": "projects/sat-io/open-datasets/CGIAR_ARIDITY", "vars": "ARID", "bands": ["b1"], "scale": 1000, "kind": "image", "start": "1970-01-01", "end": "2000-12-31", "folder": "01l_aridity_index", "usage": "Matching covariate and failure-threshold stratification.", "drive_subfolder": "01_gee_rasters/01l_aridity_index"}, "01m": {"name": "ESA WorldCover", "asset_id": "ESA/WorldCover/v200", "vars": "LC", "bands": ["Map"], "scale": 30, "kind": "image", "start": "2021-01-01", "end": "2021-12-31", "folder": "01m_worldcover", "usage": "High-resolution natural versus managed land-cover mask.", "drive_subfolder": "01_gee_rasters/01m_worldcover"}, "01n": {"name": "Landsat 8/9 Surface Reflectance", "asset_id": "LANDSAT/LC08/C02/T1_L2", "vars": "MULTI", "bands": ["SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7", "ST_B10", "QA_PIXEL", "QA_RADSAT"], "scale": 30, "kind": "collection", "start": "2013-04-01", "end": "2024-09-30", "folder": "01n_landsat", "usage": "High-resolution spectral context and optional indices.", "drive_subfolder": "01_gee_rasters/01n_landsat"}, "01o": {"name": "Sentinel-2 Harmonized", "asset_id": "COPERNICUS/S2_HARMONIZED", "vars": "MULTI", "bands": ["B1", "B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B9", "B11", "B12", "QA60", "SCL"], "scale": 30, "kind": "collection", "start": "2017-03-28", "end": "2024-09-30", "folder": "01o_sentinel2", "usage": "Recent high-resolution vegetation context.", "drive_subfolder": "01_gee_rasters/01o_sentinel2"}, "01p": {"name": "PML-V2 ET", "asset_id": "projects/sat-io/open-datasets/PML_V2", "vars": "ET", "bands": ["Ec", "Es", "Ei", "ET_water", "ET"], "scale": 1000, "kind": "collection", "start": "2000-01-01", "end": "2023-12-31", "folder": "01p_pml_v2", "usage": "Longer evaporative-response record.", "drive_subfolder": "01_gee_rasters/01p_pml_v2"}, "01q": {"name": "SSEBop ET", "asset_id": "USGS/ssebop/ssebopeta_v4", "vars": "ET", "bands": ["et"], "scale": 1000, "kind": "collection", "start": "2003-01-01", "end": "2024-09-30", "folder": "01q_ssebop", "usage": "Longer operational evaporative-response record.", "drive_subfolder": "01_gee_rasters/01q_ssebop"}, "01r": {"name": "SoilGrids", "asset_id": "projects/soilgrids-isric/", "vars": "MULTI", "bands": ["clay_0-5cm_mean", "sand_0-5cm_mean", "bdod_0-5cm_mean", "soc_0-5cm_mean"], "scale": 1000, "kind": "image", "start": "2017-01-01", "end": "2017-12-31", "folder": "01r_soilgrids", "usage": "Rooting depth, texture, and soil-water-capacity controls.", "drive_subfolder": "01_gee_rasters/01r_soilgrids"}, "01s": {"name": "NASADEM", "asset_id": "NASA/NASADEM_HGT/001", "vars": "ELEV", "bands": ["elevation", "num", "swb"], "scale": 1000, "kind": "image", "start": "2000-01-01", "end": "2000-12-31", "folder": "01s_nasadem", "usage": "Hydrologic conditioning and topographic position.", "drive_subfolder": "01_gee_rasters/01s_nasadem"}}, "independent_datasets": {"02a": {"name": "Ma et al. (2026) Water Table Depth raster", "vars": "WTD", "folder": "02a_ma_wtd_raster", "method": "requests / rioxarray", "endpoint": "REPLACE_WITH_HYDROSHARE_OR_ZENODO_URL", "usage": "Static depth-to-water axis; not a time series.", "drive_subfolder": "02_independent/02a_ma_wtd_raster"}, "02b": {"name": "USGS Groundwater Wells (NWIS)", "vars": "WELL", "folder": "02b_usgs_wells", "method": "dataretrieval", "endpoint": "https://waterservices.usgs.gov/nwis/gwlevels/", "usage": "Observed water-table decline and recovery.", "drive_subfolder": "02_independent/02b_usgs_wells"}, "02c": {"name": "State Well Networks", "vars": "WELL", "folder": "02c_state_wells", "method": "requests", "endpoint": "https://www3.twdb.texas.gov/apps/waterdatainteractive/", "usage": "State networks augment NWIS spatial coverage.", "drive_subfolder": "02_independent/02c_state_wells"}, "02d": {"name": "FLUXNET / AmeriFlux", "vars": "FLUX", "folder": "02d_ameriflux", "method": "ameriflux-api / requests", "endpoint": "REQUIRES_AMERIFLUX_ACCOUNT", "usage": "Independent site validation for buffering and decoupling.", "drive_subfolder": "02_independent/02d_ameriflux"}, "02e": {"name": "SAPFLUXNET", "vars": "SAP", "folder": "02e_sapfluxnet", "method": "requests", "endpoint": "REQUIRES_MANUAL_REGISTRATION", "usage": "Independent plant water-use validation.", "drive_subfolder": "02_independent/02e_sapfluxnet"}, "02f": {"name": "GLEAM4", "vars": "ET", "folder": "02f_gleam4", "method": "requests / xarray", "endpoint": "REQUIRES_GLEAM_REGISTRATION_URL", "usage": "Reference model product, not independent evidence.", "drive_subfolder": "02_independent/02f_gleam4"}, "02g": {"name": "GOSIF", "vars": "SIF", "folder": "02g_gosif", "method": "requests / xarray", "endpoint": "https://www.numericalterra.com/", "usage": "Long-record context and reversibility testing.", "drive_subfolder": "02_independent/02g_gosif"}, "02h": {"name": "LANID Irrigation Mapping", "vars": "IRRIG", "folder": "02h_lanid", "method": "requests / rioxarray", "endpoint": "REPLACE_WITH_ZENODO_OR_HYDROSHARE_URL", "usage": "High-resolution irrigation stratification.", "drive_subfolder": "02_independent/02h_lanid"}, "02i": {"name": "USDA NASS Statistics", "vars": "NASS", "folder": "02i_nass_stats", "method": "nass / requests", "endpoint": "https://quickstats.nass.usda.gov/api/api_GET/", "usage": "Agricultural outcome validation.", "drive_subfolder": "02_independent/02i_nass_stats"}, "02j": {"name": "ECOSTRESS Level-2 ET and LST", "vars": "MULTI", "folder": "02j_ecostress", "method": "earthaccess", "endpoint": "ECOSTRESS_L2_LSTE", "usage": "Recent high-resolution ET and LST dry-down detail.", "drive_subfolder": "02_independent/02j_ecostress"}}, "output_folders": ["01a_grace_mascons", "01b_smap_l4_rzsm", "01d_tropomi_sif", "01e_era5_land", "01f_gridmet", "01g_modis_ndvi_evi", "01h_modis_landcover", "01i_srtm_dem", "01j_usda_cdl", "01k_gfsad", "01l_aridity_index", "01m_worldcover", "01n_landsat", "01o_sentinel2", "01p_pml_v2", "01q_ssebop", "01r_soilgrids", "01s_nasadem", "01c_smap_l3_vod", "02a_ma_wtd_raster", "02b_usgs_wells", "02c_state_wells", "02d_ameriflux", "02e_sapfluxnet", "02f_gleam4", "02g_gosif", "02h_lanid", "02i_nass_stats", "02j_ecostress"]}'
(SETUP_DIR / 'config.yaml').write_text(yaml.safe_dump(json.loads(CONFIG_YAML), sort_keys=False), encoding='utf-8')
for leaf in ['01a_grace_mascons', '01b_smap_l4_rzsm', '01d_tropomi_sif', '01e_era5_land', '01f_gridmet', '01g_modis_ndvi_evi', '01h_modis_landcover', '01i_srtm_dem', '01j_usda_cdl', '01k_gfsad', '01l_aridity_index', '01m_worldcover', '01n_landsat', '01o_sentinel2', '01p_pml_v2', '01q_ssebop', '01r_soilgrids', '01s_nasadem', '01c_smap_l3_vod', '02a_ma_wtd_raster', '02b_usgs_wells', '02c_state_wells', '02d_ameriflux', '02e_sapfluxnet', '02f_gleam4', '02g_gosif', '02h_lanid', '02i_nass_stats', '02j_ecostress']:
    ensure_layout(DRIVE_ROOT, ('01_gee_rasters/' if leaf.startswith('01') else '02_independent/') + leaf)
print('AOI and Drive tree ready:', DRIVE_ROOT)


AOI and Drive tree ready: Ogallala_Phase0


In [3]:
logger, LOG_PATH = configure_logger('00_setup', DRIVE_ROOT)


In [4]:
DATASET = {'dataset_name': 'Ogallala Aquifer Boundary', 'source': 'USGS', 'access_method': 'requests / geopandas'}


In [5]:
# The setup cell above is the dataset definition and AOI processing pipeline.


In [6]:
# Setup has no XEE pull; it writes the serialized AOI handoff.


In [7]:
# No server-side export is needed for the vector AOI.


In [8]:
%pip -q install compliance-checker
MAX_CONCURRENT = 20
MAX_DAILY_TASKS = 2500


Note: you may need to restart the kernel to use updated packages.


In [9]:
# CF writing is provided by 00_setup/utils.py for downstream notebooks.


In [10]:
# The setup manifest and file checks are emitted in the final cell.


In [11]:
required = [SETUP_DIR / name for name in ['ogallala_boundary.geojson', 'ogallala_bbox.json', 'aoi_ee_geometry.json', 'config.yaml', 'utils.py']]
write_manifest(DRIVE_ROOT, '00_setup', required, 'Ogallala Aquifer Boundary')
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise RuntimeError(f'Missing setup outputs: {missing}')
print((SETUP_DIR / 'config.yaml').read_text(encoding='utf-8'))
print('Setup complete:', len(required), 'files')


DRY_RUN: false
GEE_PROJECT: ee-ishansinhagzb
MASTER_CRS: EPSG:5070
MASTER_SCALE:
  01a: 1000
  01b: 1000
  01d: 1000
  01e: 1000
  01f: 1000
  01g: 1000
  01h: 1000
  01i: 1000
  01j: 30
  01k: 1000
  01l: 1000
  01m: 30
  01n: 30
  01o: 30
  01p: 1000
  01q: 1000
  01r: 1000
  01s: 1000
AOI_BUFFER_METERS: 1000
START_DATE: '2000-01-01'
END_DATE: '2024-09-30'
datasets:
  01a:
    name: GRACE / GRACE-FO Mascons
    asset_id: NASA/GRACE/MASS_GRIDS_V04/MASCON
    vars: TWSA
    bands:
    - lwe_thickness
    scale: 1000
    kind: collection
    start: '2002-04-01'
    end: '2024-09-30'
    folder: 01a_grace_mascons
    usage: Basin-scale storage constraint and slow driver; use for closure and screening.
    drive_subfolder: 01_gee_rasters/01a_grace_mascons
  01b:
    name: SMAP L4 Root Zone Soil Moisture and L3 VOD
    asset_id:
    - NASA/SMAP/SPL4SMGP/008
    - NASA/SMAP/SPL3SMP_E/005
    vars: MULTI
    bands:
    - sm_rootzone
    - vegetation_water_content
    scale: 1000
    kind: co